# 08 — RAG Evaluation: Retrieval Precision/Recall and LLM Stance-Label Spot-Check

Before this notebook, retrieval and generation quality were judged by eyeballing individual examples (see [decoded-state-to-text-report.md](../docs/decoded-state-to-text-report.md)). This builds a small, honestly-labeled evaluation set — reading each of the 6 corpus papers to make real relevance judgments, not guessing — and measures:

1. **Retrieval precision/recall** at k=5, dense-only vs. dense+cross-encoder-rerank.
2. **LLM stance-label spot-check** — does `pipeline.py`'s SUPPORTS/CONTRADICTS/UNRELATED labeling match manual judgment on a couple of examples.

**Caveat up front**: 5 near-identical queries (one per MOTOR condition) against 6 papers is a small, first-pass evaluation, not a rigorous benchmark — good enough to catch an obviously broken reranker or a wildly hallucinating LLM, not to certify quality.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurolens.retrieval import load_index, load_embedding_model, load_reranker, retrieve_chunks, retrieve_and_rerank
from neurolens.pipeline import build_query_text

chunks, embeddings = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
embedding_model = load_embedding_model()
reranker = load_reranker()
print(f"{len(chunks)} chunks from {len(set(c.source_file for c in chunks))} papers")


/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13141.09it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7338.69it/s]

879 chunks from 8 papers


## 1. Gold relevance judgments (read, not guessed)

Read page 1 of every paper in the corpus to ground these judgments in actual content, not filenames. **Updated**: the corpus grew from 6 to 8 papers after this eval was first written (Ehrsson 2003 and Meier 2008 were added later) - the gold set below was not updated at the time, which silently made the eval measure retrieval against a stale, narrower notion of "relevant" than the corpus the pipeline actually searches. Fixed here.

- **Ehrsson et al. 2003** (`ehrsson-et-al-2003-...pdf`) - imagery of finger/toe/tongue movement activates body-part-specific motor cortex. **HIGH relevance** - the single most on-topic paper in the corpus for any hand/foot/tongue query, more specific than the atlas-definition papers below.
- **Meier et al. 2008** (`meier-et-al-2008-...pdf`) - high-resolution fMRI of primary motor cortex organization, non-classical/overlapping somatotopy. **HIGH relevance** for the same reason.
- **Barch et al. 2013** (`1-s2.0-S1053811913005272-main.pdf`) - the actual HCP task-fMRI design paper, explicitly describes the MOTOR task (hand/foot/tongue movements). **HIGH relevance** to any MOTOR-condition query.
- **Yeo et al. 2011** (`thomas-yeo-et-al-2011...pdf`) - defines the 7-network parcellation (SomMot, Default, etc.) named in every query. **Relevant** for grounding the RSN vocabulary, not the task itself.
- **Schaefer et al. 2018** (`bhx179.pdf`) - defines the exact 300-ROI atlas and network assignments this pipeline uses. **Relevant** for the same reason as Yeo 2011.
- **Van Essen et al. 2013** (`1-s2.0-S1053811913005351-main.pdf`) - general HCP project overview, doesn't discuss MOTOR-task specifics or RSNs in depth. **Background relevance only.**
- **van den Heuvel & Sporns 2013** (`PIIS1364661313002167.pdf`) - "Network hubs in the human brain," a connectome graph-theory review unrelated to task decoding or the Yeo/Schaefer network vocabulary. **Not relevant** to these queries.
- **Misra & Surampudi et al. 2021** (`journal.pcbi.1008943.pdf`) - GRU-based decoding of naturalistic movie-viewing fMRI, a different task domain entirely. **Not relevant** to MOTOR-condition queries (methodologically adjacent, topically off).

**Relevant set for all 5 queries below**: `{Ehrsson2003, Meier2008, Barch2013, Yeo2011, Schaefer2018}` (5 of 8 papers).


In [2]:
RELEVANT_PAPERS = {
    "ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf",  # Ehrsson 2003
    "meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf",  # Meier 2008
    "1-s2.0-S1053811913005272-main.pdf",  # Barch 2013
    "thomas-yeo-et-al-2011-the-organization-of-the-human-cerebral-cortex-estimated-by-intrinsic-functional-connectivity.pdf",  # Yeo 2011
    "bhx179.pdf",  # Schaefer 2018
}
N_RELEVANT_PAPERS = len(RELEVANT_PAPERS)

CONDITIONS = ["left_hand", "right_hand", "left_foot", "right_foot", "tongue"]

def fake_rsn_attribution(network: str) -> dict:
    return {"consensus_network": network, "families_agree": True, "top_network_by_method": {"saliency": network}}

eval_queries = [
    build_query_text(condition, 0.95, fake_rsn_attribution("SomMot"), subject_id="101915", task="MOTOR", run="LR")
    for condition in CONDITIONS
]
for q in eval_queries:
    print(q)


Decoded condition: left_hand (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: right_hand (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: left_foot (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: right_foot (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.
Decoded condition: tongue (confidence 95%). Primary contributing resting-state network: SomMot. MOTOR task, subject 101915, run LR.


## 2. Precision/recall at k=5: dense-only vs. dense+rerank

In [3]:
def precision_recall(retrieved_df: pd.DataFrame) -> dict:
    retrieved_papers = set(retrieved_df["source_file"])
    hit_papers = retrieved_papers & RELEVANT_PAPERS
    n_relevant_chunks = retrieved_df["source_file"].isin(RELEVANT_PAPERS).sum()
    precision = n_relevant_chunks / len(retrieved_df)
    recall = len(hit_papers) / N_RELEVANT_PAPERS
    # precision is chunk-level (of the 5 retrieved chunks) and recall is paper-level
    # (of the N relevant papers) - different units, so F1 here is a summary of two
    # complementary signals, not a textbook single-population F1. Still useful: it
    # penalizes a method that gets one right but not the other, which neither number
    # alone does.
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {
        "precision_at_5": precision,
        "recall_papers_at_5": recall,
        "f1_at_5": f1,
    }


rows = []
for query, condition in zip(eval_queries, CONDITIONS):
    dense = retrieve_chunks(query, model=embedding_model, embeddings=embeddings, chunks=chunks, top_k=5)
    reranked = retrieve_and_rerank(query, model=embedding_model, embeddings=embeddings, chunks=chunks,
                                     reranker=reranker, candidate_k=20, top_k=5)
    dense_metrics = precision_recall(dense)
    rerank_metrics = precision_recall(reranked)
    rows.append({"condition": condition, "method": "dense-only", **dense_metrics})
    rows.append({"condition": condition, "method": "dense+rerank", **rerank_metrics})

eval_df = pd.DataFrame(rows)
eval_df


,condition,method,precision_at_5,recall_papers_at_5,f1_at_5
0,left_hand,dense-only,1.0,0.6,0.750000
1,left_hand,dense+rerank,1.0,0.4,0.571429
2,right_hand,dense-only,1.0,0.6,0.750000
3,right_hand,dense+rerank,1.0,0.4,0.571429
4,left_foot,dense-only,1.0,0.2,0.333333
5,left_foot,dense+rerank,1.0,0.4,0.571429
6,right_foot,dense-only,1.0,0.4,0.571429
7,right_foot,dense+rerank,1.0,0.4,0.571429
8,tongue,dense-only,1.0,0.6,0.750000
9,tongue,dense+rerank,1.0,0.6,0.750000


In [4]:
summary = eval_df.groupby("method")[["precision_at_5", "recall_papers_at_5"]].mean()
print("Mean across all 5 queries:")
summary


Mean across all 5 queries:


,precision_at_5,recall_papers_at_5
method,,
dense+rerank,1.0,0.44
dense-only,1.0,0.48


## 3. LLM stance-label spot-check (2 examples)

Manually judged against the same gold relevance set above. Real LLM calls (~20s each), kept to 2 examples for wall-clock reasons — genuinely just a spot-check, not a full evaluation.


In [5]:
from neurolens.pipeline import build_llm_prompt, make_mlx_generate_fn

generate_fn = make_mlx_generate_fn()

spot_check_rows = []
for query, condition in zip(eval_queries[:2], CONDITIONS[:2]):
    retrieved = retrieve_chunks(query, model=embedding_model, embeddings=embeddings, chunks=chunks, top_k=5).to_dict(orient="records")
    prompt = build_llm_prompt(query, retrieved)
    t0 = time.time()
    generated = generate_fn(prompt)
    elapsed = time.time() - t0
    print(f"=== {condition} (generated in {elapsed:.1f}s) ===")
    for i, r in enumerate(retrieved, start=1):
        gold = "RELEVANT" if r["source_file"] in RELEVANT_PAPERS else "not relevant"
        print(f"  [{i}] {r['source_file']} (gold: {gold})")
    print()
    print(generated)
    print()
    spot_check_rows.append({"condition": condition, "retrieved": retrieved, "generated_text": generated})


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 3585.90it/s]

=== left_hand (generated in 14.8s) ===
  [1] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)
  [2] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)
  [3] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)
  [4] meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf (gold: RELEVANT)
  [5] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)

1. Excerpt 1: SUPPORTS the decoded result, as it discusses the activation of the working memory task and the Edinburgh Handedness questionnaire, which is relevant to the decoded condition of "left_hand".
2. Excerpt 2: SUPPORTS the decoded result, as it discusses the activation of motor imagery of various types of hand actions, which is rel

=== right_hand (generated in 15.0s) ===
  [1] 1-s2.0-S1053811913005272-main.pdf (gold: RELEVANT)
  [2] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)
  [3] meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf (gold: RELEVANT)
  [4] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)
  [5] ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf (gold: RELEVANT)

Here are the analyses for each excerpt:

1. Excerpt 1: SUPPORTS
This excerpt discusses the activation of the working memory task, which is related to the decoded result of the motor task. The activation maps show that the motor task activates areas associated with sensorimotor control, which is consistent with the decoded result.

Synthes

## Summary

**Correction applied this run**: the gold relevance set above was written for the original 6-paper corpus (3 relevant) and never updated after Ehrsson 2003 and Meier 2008 were added later, making the corpus 8 papers with 5 relevant. Re-running against the corrected gold set changed the numbers - this summary reflects the corrected run, not the original.

**Retrieval (5 queries, dense-only vs. dense+cross-encoder-rerank, k=5), corrected gold set:**

| Method | Precision@5 | Recall (relevant papers)@5 | F1@5 |
|---|---|---|---|
| Dense-only | 1.00 | **0.48** | **0.63** |
| Dense+rerank | 1.00 | 0.44 | 0.61 |

Precision is still perfect - every top-5 chunk comes from *a* relevant paper either way, at this corpus size. Recall is now genuinely informative (5 relevant papers to find, not 3) and reveals real room to improve. The previously reported reranking benefit (recall 0.80 -> 1.00) does not survive the correction: under the correct gold set, reranking is marginally *behind* dense-only on both recall and F1 - within noise at n=5 queries, but a real reversal of the earlier conclusion, not a rounding difference. Kept enabled by default anyway (near-zero cost); its value is expected to reappear once the corpus is large enough that dense retrieval's precision stops being trivially perfect.

**LLM stance-label spot-check (2 examples):** both `left_hand` and `right_hand` retrieved 5/5 genuinely relevant chunks (mostly from Ehrsson 2003 and Meier 2008, the two papers added later - direct evidence the corpus expansion helped) and the model labeled nearly all of them SUPPORTS, which is coarsely correct. But the *justifications* stay generic rather than precise - e.g. citing an excerpt's mention of "the working memory task and the Edinburgh Handedness questionnaire" as support for a hand-movement decode, when the excerpt's actual relevance is that it's from the HCP task-design paper, not that specific unrelated detail. Consistent with the standing finding: coarse relevance judgments are more trustworthy than the natural-language reasoning behind them.

**Takeaway**: retrieval precision is solid and measured, not assumed, at this corpus size; recall has real, now-visible room to improve, and the reranker's benefit is unproven rather than confirmed. Generation quality remains the more fragile piece - treat this pipeline's generated text as a retrieval-grounded *draft*, not a verified explanation.

Findings folded into [docs/case1-summary-report.md](../docs/case1-summary-report.md) §8.
